# Set Up + Imports

In [1]:
from collections import OrderedDict
from typing import Optional
from typing import OrderedDict
from tqdm import tqdm
import sys
import json
import numpy as np
from numpy.random import Generator
from pydantic import BaseModel
from pydantic import ConfigDict
from pydantic import Field


import networkx as nx
import enum

from torch.utils.data import Subset
from torch.utils.data import ConcatDataset
from torchvision import transforms
from torchvision import models
from torchvision.models import ResNet18_Weights # Import weights enum
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Subset, random_split, Dataset
from torch.utils.data import DataLoader
import torch

from transformers import AutoTokenizer
from transformers import BertForSequenceClassification
from torch.optim import AdamW
from torch.optim import SGD
import torch.nn as nn
import torch.optim as optim


from wilds import get_dataset
from sklearn.metrics import f1_score, classification_report

# Data

## Tumor Classification

In [ ]:
full_dataset = get_dataset(dataset="camelyon17", download=True)

class TransformedSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        # Get the raw PIL image, label, and metadata from the subset
        x, y, metadata = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y, metadata

    def __len__(self):
        return len(self.subset)

# Use this in your get_manual_splits function
def get_manual_splits(dataset, train_ratio=0.2, transform=None):
    hospital_ids = dataset.metadata_array[:, 0]
    indices_0123 = torch.where(hospital_ids <= 3)[0]
    indices_4 = torch.where(hospital_ids == 4)[0]

    def split_indices(indices, ratio):
        train_size = int(len(indices) * ratio)
        return random_split(indices, [train_size, len(indices) - train_size])

    # Get the raw subsets
    raw_train_0123, raw_test_0123 = split_indices(indices_0123, train_ratio)
    raw_train_4, raw_test_4 = split_indices(indices_4, train_ratio)

    # Wrap them to apply the transform
    return (
        TransformedSubset(Subset(dataset, raw_train_0123), transform),
        TransformedSubset(Subset(dataset, raw_test_0123), transform),
        TransformedSubset(Subset(dataset, raw_train_4), transform),
        TransformedSubset(Subset(dataset, raw_test_4), transform)
    )

transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(), # This converts PIL -> Tensor
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Create all 4 subsets
train_0123, test_0123, train_4, test_4 = get_manual_splits(full_dataset, transform=transform)

print(f"Hospitals 0-3 -> Train: {len(train_0123)}, Test: {len(test_0123)}")
print(f"Hospital 4    -> Train: {len(train_4)}, Test: {len(test_4)}")

# Define batch size
BATCH_SIZE = 256

train_loader_0123 = DataLoader(train_0123, batch_size=BATCH_SIZE, shuffle=True)
train_loader_4 = DataLoader(train_4, batch_size=BATCH_SIZE, shuffle=True)


test_loader_0123 = DataLoader(test_0123, batch_size=BATCH_SIZE, shuffle=False)
test_loader_4 = DataLoader(test_4, batch_size=BATCH_SIZE, shuffle=False)

You can also download the dataset manually at https://wilds.stanford.edu/downloads.


10658897920Byte [02:55, 60649355.11Byte/s]                               


Extracting data/camelyon17_v1.0/archive.tar.gz to data/camelyon17_v1.0


In [ ]:
def resnet18_camelyon17(num_classes=2, use_pretrained=True):
    torch.manual_seed(2809)

    # Use ImageNet weights if use_pretrained is True
    weights = ResNet18_Weights.IMAGENET1K_V1 if use_pretrained else None
    m = models.resnet18(weights=weights)

    # Replace the final layer for the binary task
    m.fc = torch.nn.Linear(m.fc.in_features, num_classes)

    return m

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#model = TumorModule(num_classes=2).to(device)
model = resnet18_camelyon17(num_classes=2, use_pretrained=True).to(device)
optimizer = optim.SGD(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels, _ in tqdm(loader):
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Stats
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

def evaluate(model, loader, name="Test Set"):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels, _ in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    accuracy = 100. * correct / total
    print(f"{name} Accuracy: {accuracy:.2f}%")
    return accuracy

num_epochs = 5

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader_0123, optimizer, criterion)
    print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")

    # Evaluate on both test sets
    evaluate(model, test_loader_0123, name="Hospitals 0-3 (Seen)")
    evaluate(model, test_loader_4, name="Hospital 4 (Unseen)")
    print("-" * 30)


## Civil Comments

In [ ]:
# Load the full dataset (downloads if not exists)
dataset = get_dataset(dataset=DATASET_NAME, root_dir=ROOT_DIR, download=True)


SUBSET_SIZE = 3000


# Using range(SUBSET_SIZE) gets the first 1000 samples
train_data = Subset(dataset.get_subset("train"), range(SUBSET_SIZE))
val_data   = Subset(dataset.get_subset("val"),   range(SUBSET_SIZE))
test_data  = Subset(dataset.get_subset("test"),  range(SUBSET_SIZE))

In [ ]:


# Load a pre-trained tokenizer (e.g., BERT)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
MAX_LENGTH = 512 # Or adjust as needed

def collate_fn_with_tokenizer(batch):
    # 'batch' is a list of (text, label, metadata) tuples from the dataset
    texts = [item[0] for item in batch]
    labels = torch.tensor([item[1] for item in batch], dtype=torch.long)
    metadata = torch.stack([item[2] for item in batch]) # Metadata can also be useful for group-wise evaluation

    # Tokenize and pad the texts
    encoded_texts = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )
    return encoded_texts, labels, metadata

# Create DataLoaders
BATCH_SIZE = 16

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, collate_fn=collate_fn_with_tokenizer)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, collate_fn=collate_fn_with_tokenizer)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, collate_fn=collate_fn_with_tokenizer)


In [ ]:
# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load a pre-trained BERT model for sequence classification
# The Civil Comments dataset is binary classification (toxic vs non-toxic)
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
model.to(device)

# Define loss function and optimizer
weights = torch.tensor([3.0, 1.0]).to(device) # Non-toxic is 3x rarer, so it gets 3x weight

# 2. Initialize the weighted loss function
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = AdamW(model.parameters(), lr=5e-5) # AdamW is a common optimizer for transformers

# Training loop (simplified)
NUM_EPOCHS = 1

for epoch in range(NUM_EPOCHS):
    model.train()
    for batch in tqdm(train_loader):
        inputs, labels, metadata = batch
        inputs = {k: v.to(device) for k, v in inputs.items()}
        labels = labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(**inputs, labels=labels)
        logits = outputs.logits

        loss = criterion(logits, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} finished. Loss: {loss.item():.4f}")

print("Finished Training")

In [ ]:
all_preds = []
all_labels = []
all_metadata = []

model.eval()
with torch.no_grad():
    for batch in test_loader:
        inputs, labels, metadata = batch
        inputs = {k: v.to(device) for k, v in inputs.items()}

        outputs = model(**inputs)
        preds = torch.argmax(outputs.logits, dim=1)

        # Collect results back to CPU
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())
        all_metadata.append(metadata.cpu())

# Concatenate all batches into single tensors
all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)
all_metadata = torch.cat(all_metadata)

y_true = all_labels.numpy()
y_pred = all_preds.numpy()


print(f"F1-Score: {f1:.4f}")

# Comprehensive report including precision and recall
print(classification_report(y_true, y_pred, target_names=['Non-Toxic', 'Toxic']))